# Training Analysis

Explore local training runs, health summaries, hyperparameter effects, and metric curves using the shared `vla.analysis` helpers.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import seaborn as sns

from vla.analysis import (
    annotate_cohort_health,
    build_long_metric_table,
    build_rollout_success_percent_table,
    build_run_summaries,
    build_runs_table,
    group_summaries_by_cohort,
    load_training_runs,
    rank_run_summaries,
    summarize_hyperparameter_effects,
    to_dataframe,
)

sns.set_theme(style="whitegrid")

In [ ]:
runs = load_training_runs(ROOT / "results")
summaries = annotate_cohort_health(build_run_summaries(runs))
run_df = to_dataframe(build_runs_table(summaries))
hp_df = to_dataframe(summarize_hyperparameter_effects(summaries))
cohorts = group_summaries_by_cohort(summaries)

print(f"runs={len(runs)} cohorts={len(cohorts)}")
run_df[["name", "method", "update_method", "num_tasks", "final_eval", "best_eval", "success_drop", "healthy"]].sort_values(
    ["final_eval", "success_drop"], ascending=[False, True]
).head(15)

In [ ]:
healthy_df = run_df[run_df["healthy"]].sort_values(["final_eval", "success_drop"], ascending=[False, True])
healthy_df[[
    "name",
    "num_tasks",
    "task_ids",
    "final_eval",
    "best_eval",
    "success_drop",
    "lr",
    "sft_kl_coeff",
    "include_demos_in_update",
    "success_replay_total_size",
]].head(20)

In [ ]:
multitask_df = run_df[(run_df["num_tasks"] > 1) & (run_df["method"] == "sparse_rl")].copy()
multitask_df[[
    "name",
    "task_ids",
    "final_eval",
    "best_eval",
    "success_drop",
    "healthy",
    "sft_kl_coeff",
    "include_demos_in_update",
    "success_replay_total_size",
]].sort_values(["final_eval", "success_drop"], ascending=[False, True])

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=multitask_df,
    x="final_eval",
    y="success_drop",
    hue="sft_kl_coeff",
    style="include_demos_in_update",
    size="success_replay_total_size",
    sizes=(60, 250),
)
plt.title("Multitask Runs: Final Eval vs Degradation")
plt.xlabel("Final Eval Success Rate")
plt.ylabel("Best - Final")
plt.show()

In [ ]:
target_runs = [
    run for run in runs
    if run.name in {
        "v23-t_multi_sparse_rl_5tasks_spatial_seed42",
        "v29-t_multitask-sft-kl-0.0025_sparse_rl_5tasks_spatial_seed42",
        "v31-t_0,2,5,7,9-sft-kl-0.005_sparse_rl_5tasks_spatial_seed42",
    }
]
curve_df = to_dataframe(build_long_metric_table(target_runs, "sparse_rl/eval/success_rate", dedupe_x=True))
curve_df = curve_df.rename(columns={"x": "iteration"})

plt.figure(figsize=(9, 5))
sns.lineplot(data=curve_df, x="iteration", y="value", hue="name", marker="o")
plt.title("Comparable Multitask Cohort: Eval Success Curves")
plt.ylabel("Eval Success Rate")
plt.show()

In [ ]:
cohort_runs = [
    run for run in runs
    if run.task_ids == (
        "spatial_task_0",
        "spatial_task_2",
        "spatial_task_5",
        "spatial_task_7",
        "spatial_task_9",
    )
]
success_df = to_dataframe(build_rollout_success_percent_table(cohort_runs))
success_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
for run_name, group in success_df.groupby("name"):
    group = group.sort_values("iteration")
    alpha = float(group["run_order_alpha"].iloc[0])
    order = int(group["run_order"].iloc[0])
    plt.plot(
        group["iteration"],
        group["success_pct"],
        label=f"{order}: {run_name}",
        alpha=alpha,
        linewidth=2,
    )
plt.title("Rollout Success % Per Iteration (alpha increases with run recency)")
plt.xlabel("Iteration")
plt.ylabel("Rollout successes / total rollout trajectories [%]")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
hp_df[hp_df["cohort_label"].str.contains("spatial_task_0, spatial_task_2, spatial_task_5, spatial_task_7, spatial_task_9", na=False)].sort_values(
    ["hyperparameter", "mean_final_eval"], ascending=[True, False]
)